# bsc_01 — M0 Headroom  ⭐ **GATE 1 (cổng chặn chính)**

**Đây là thứ quan trọng nhất của Stage 1.** Không có trong plan doc. Nó hỏi đúng câu
ROI cascade đã quên: **phần thưởng lớn cỡ nào — TRƯỚC khi xây bất cứ thứ gì.**

Chạy trên **OAI-ZIB test (103 ca)** — nguồn duy nhất có GT xương. ~1h CPU, **không GPU**.

Đo, với mỗi ca × mỗi lớp sụn:
1. Trường độ dày GT (từ tia dọc pháp tuyến xương).
2. Phân rã **error mass** của ResEnc (B0) theo bin độ dày — hai chiều.
3. Counterfactual: nếu triệt tiêu lỗi ở bin {absent, ≤0.5, ≤1.0}, ASSD cải thiện bao nhiêu.
4. M0c: bậc thang z của GT.

**GATE 1:**
- ✅ PROCEED: `ΔASSD_prize ≥ 0.08mm` **và** thin+absent ≥ 35% error mass
- ⚠️ RESCOPE: `ΔASSD_prize ∈ [0.04, 0.08)` → viết lại mục tiêu quanh presence/thickness
- ❌ STOP: `ΔASSD_prize < 0.04mm` **hoặc** bậc thang z chiếm ưu thế

**Xác suất thật thà: 50/50.** Sụn đùi phần lớn dày 2–3mm. Nhưng error mass (không phải
diện tích) mới đáng kể — vùng mỏng khó bất tương xứng.


### Cell config (giống bsc_00)

In [ ]:
# ============================================================
# CELL CONFIG CHUAN - tai dung o MOI notebook bsc_*
# Drive-first: MOI artifact nam duoi BSC_ROOT. KHONG ghi vao /content/.
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

REPO_URL = "https://github.com/<user>/nnUnet-OAI"   # <-- doi thanh repo cua ban
REPO_DIR = "/content/repo"

import os, sys
if not os.path.isdir(REPO_DIR):
    !git clone -q $REPO_URL $REPO_DIR
!cd $REPO_DIR && git pull -q
sys.path.insert(0, REPO_DIR)

# Thu can cho Colab (may local da co scipy/skimage/numpy)
!pip install -q nibabel SimpleITK 2>/dev/null

BSC_ROOT = "/content/drive/MyDrive/bsc"          # goc artifact - TAT CA nam duoi day
os.makedirs(BSC_ROOT, exist_ok=True)
for sub in ["splits","baselines","geom","raydb","atlas","runs"]:
    os.makedirs(f"{BSC_ROOT}/{sub}", exist_ok=True)

# Duong du lieu cu (READ-ONLY - khong bao gio ghi de)
RAW = "/content/drive/MyDrive/nnUNet_raw/Dataset001_KneeOA"   # <-- kiem lai duong nay
print("BSC_ROOT =", BSC_ROOT)
print("Cach ly: doc RAW read-only, ghi MOI THU duoi BSC_ROOT, dataset/folder moi.")

## Chạy M0 trên 103 ca test

Cần **prediction B0 trên test set** (không phải CV). Nếu chưa có, chạy inference d020
fold_0 trên `imagesTs` (103 ca) rồi lưu vào `BSC_ROOT/baselines/B0_test_pred/`.
Đây là lần **duy nhất** dùng test set trước Gate 4 — chỉ để đo headroom, không tune gì.

In [ ]:
# ---- Sinh B0_test_pred neu chua co: chay B0 (250ep fold_0) tren 103 anh test ----
# Day la buoc THIEU trong notebook goc: markdown cell tren mo ta nhung khong co code.
# B0_test_pred KHONG nam trong zip (zip chi co CV validation pred), phai TU chay inference.
# ⭐ BUOC NAY DUNG GPU - bat runtime GPU cho nhanh (khac vong metric CPU o Gate 0/M0).
import os, glob
B0_PRED = f"{BSC_ROOT}/baselines/B0_test_pred"
os.makedirs(B0_PRED, exist_ok=True)

n_img  = len(glob.glob(f"{RAW}/imagesTs/*_0000.nii.gz"))
n_have = len(glob.glob(f"{B0_PRED}/oaizib_*.nii.gz"))
print(f"Anh test: {n_img} | prediction da co: {n_have}")
assert n_img > 0, f"Khong thay anh test o {RAW}/imagesTs (can *_0000.nii.gz)"

if n_have < n_img:
    # nnUNet doc model tu $nnUNet_results. Zip da bung full folder trainer vao ds020.
    os.environ["nnUNet_results"]      = f"{BSC_ROOT}/baselines/ds020"
    os.environ["nnUNet_raw"]          = os.path.dirname(RAW)        # nnUNet doi co, khong dung khi predict
    os.environ["nnUNet_preprocessed"] = "/content/nnunet_prep_tmp"  # tmp, khong dung khi predict
    os.makedirs(os.environ["nnUNet_preprocessed"], exist_ok=True)

    # Model folder phai du dataset.json + plans.json + checkpoint truoc khi chay
    mdir = glob.glob(f"{os.environ['nnUNet_results']}/Dataset020_KneeUnion/"
                     f"nnUNetTrainer_250epochs*ResEnc*3d_fullres")[0]
    for need in ["dataset.json", "plans.json", "fold_0/checkpoint_best.pth"]:
        assert os.path.exists(f"{mdir}/{need}"), f"Thieu {need} trong {mdir}"
    print("Model OK:", os.path.basename(mdir))

    import importlib.util
    if importlib.util.find_spec("nnunetv2") is None:
        !pip install -q nnunetv2

    import torch
    dev = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (rat cham - bat GPU!)"
    print("Device:", dev)

    # -tr/-p/-c lay tu ten folder: nnUNetTrainer_250epochs__nnUNetResEncUNetLPlans__3d_fullres
    !nnUNetv2_predict -i "{RAW}/imagesTs" -o "{B0_PRED}" -d 020 -c 3d_fullres -tr nnUNetTrainer_250epochs -p nnUNetResEncUNetLPlans -f 0 -chk checkpoint_best.pth

    n_have = len(glob.glob(f"{B0_PRED}/oaizib_*.nii.gz"))

assert n_have >= n_img, f"Inference chua du: {n_have}/{n_img}. Xem log nnUNet ben tren."
print(f"B0_test_pred san sang: {n_have} prediction o {B0_PRED}")

In [ ]:
import glob, json, os
import numpy as np
from tqdm import tqdm
from bsc import io_utils, headroom, core
from bsc.core import RayConfig

RAW_TS_IMG = f"{RAW}/imagesTs"          # 103 anh test
RAW_TS_LAB = f"{RAW}/labelsTs"          # 103 GT test (co xuong+sun, [0..5])
B0_PRED    = f"{BSC_ROOT}/baselines/B0_test_pred"
CKPT       = f"{BSC_ROOT}/runs/M0_percase.jsonl"   # Drive-first: resume khi dut ket noi

CART = {"femoral_cart": 2, "med_tib_cart": 4}   # sun dui + chay trong (§3.1)
BONE = {"femoral_cart": 1, "med_tib_cart": 3}   # xuong tuong ung lam neo he toa do

cfg = RayConfig()
cases = io_utils.list_cases(RAW_TS_IMG)
print(f"{len(cases)} ca test")

# --- Resume: doc ket qua per-case da co ---------------------------------------
os.makedirs(os.path.dirname(CKPT), exist_ok=True)
results = {c: {"prize": [], "mass": [], "stair": []} for c in CART}
done = set()
if os.path.exists(CKPT):
    with open(CKPT) as f:
        for line in f:
            r = json.loads(line)
            done.add((r["case"], r["cls"]))
            results[r["cls"]]["prize"].append(r["prize"])
            results[r["cls"]]["mass"].append(r["mass"])
            results[r["cls"]]["stair"].append(r["stair"])
    print(f"Tiep tuc: da co {len(done)} (ca,lop) trong {CKPT}")

def _clean(o):   # numpy -> python cho json
    if isinstance(o, dict):  return {k: _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_clean(x) for x in o]
    if isinstance(o, (np.floating, np.integer)): return float(o)
    return o

n_skip = 0
with open(CKPT, "a") as fh:
    for cid in tqdm(cases, desc="M0"):
        prf = f"{B0_PRED}/{cid}.nii.gz"
        if not os.path.exists(prf):
            n_skip += 1
            continue
        need = [c for c in CART if (cid, c) not in done]
        if not need:
            continue
        gt, sp = io_utils.load_nii(f"{RAW_TS_LAB}/{cid}.nii.gz")
        pr, _ = io_utils.load_nii(prf)
        for c in need:
            bone = (gt == BONE[c])
            gt_c, pr_c = (gt == CART[c]), (pr == CART[c])
            if not gt_c.any() or not bone.any():
                continue
            # Thickness field tinh MOT lan, dung cho ca prize lan error_mass (bo marching-cubes 2x)
            tf = headroom.gt_thickness_per_node(bone, gt_c, sp, cfg)
            row = {
                "case": cid, "cls": c,
                "prize": _clean(headroom.prize_counterfactual(gt_c, pr_c, bone, sp, cfg, thickness=tf)),
                "mass":  _clean(headroom.error_mass_by_thickness(gt_c, pr_c, bone, sp, cfg, thickness=tf)),
                "stair": _clean(headroom.gt_staircase_z_vs_inplane(gt_c, sp)),
            }
            results[c]["prize"].append(row["prize"])
            results[c]["mass"].append(row["mass"])
            results[c]["stair"].append(row["stair"])
            fh.write(json.dumps(row) + "\n")
            fh.flush()   # ghi ngay: dut ket noi van con du lieu

if n_skip:
    print(f"CANH BAO: {n_skip} ca thieu prediction o {B0_PRED} (chay lai cell inference?)")
assert any(results[c]["prize"] for c in CART), (
    f"Khong tinh duoc ca nao. Thieu pred: {n_skip}. Kiem B0_test_pred va labelsTs."
)
print(f"Xong M0. Per-case da ghi: {CKPT}")

## GATE 1 — quyết định

In [ ]:
for c in CART:
    g = headroom.evaluate_gate1(results[c]["prize"], results[c]["mass"])
    print(f"\n=== {c} ===")
    print(f"  ΔASSD_prize = {g['d_assd_prize_mean_mm']:.4f} mm  CI {g['d_assd_prize_ci']}")
    print(f"  thin+absent error mass = {g['thin_absent_error_mass_frac_mean']:.1%}")
    print(f"  QUYET DINH: {g['decision']}")

    # Bang error mass theo bin
    import numpy as np
    print(f"  {'bin':<10}{'N':>8}{'mean_d':>9}{'mass%':>8}")
    for nm in core.THICKNESS_NAMES:
        fr = np.mean([m['per_bin'][nm]['frac_error_mass'] for m in results[c]['mass']])
        nn = np.mean([m['per_bin'][nm]['n'] for m in results[c]['mass']])
        md_ = np.mean([m['per_bin'][nm]['mean_dist_mm'] for m in results[c]['mass']])
        print(f"  {nm:<10}{nn:>8.0f}{md_:>9.3f}{fr:>7.1%}")

    stair = np.nanmean([s['z_over_inplane_std'] for s in results[c]['stair']])
    print(f"  M0c bac thang z/in-plane = {stair:.3f}  (>>1 => nhieu z chiem uu the)")

## Ghi kết quả M0 vào Drive (§7)

Dù quyết định là gì, **lưu lại** — kết quả âm tính M0 vẫn publishable (cùng ROI cascade).

In [ ]:
import numpy as np
now = None   # notebook: co the dat chuoi thoi gian thu cong neu muon
out = {c: {"gate1": headroom.evaluate_gate1(results[c]["prize"], results[c]["mass"])}
       for c in CART}
# ep numpy -> float cho json
def clean(o):
    if isinstance(o, dict): return {k:clean(v) for k,v in o.items()}
    if isinstance(o, (list,tuple)): return [clean(x) for x in o]
    if isinstance(o, (np.floating,np.integer)): return float(o)
    return o
path = f"{BSC_ROOT}/runs/M0_headroom_B0_ zibTs.json".replace(" ","")
json.dump(clean(out), open(path,"w"), indent=2)
print("Da ghi", path)
print("\nNEU PROCEED -> Phase 2 (geometry QC: M3/M2/M4 tren xuong that).")
print("NEU RESCOPE  -> viet lai muc tieu §3.7 quanh presence F1 + thickness MAE.")
print("NEU STOP     -> cong bo ket qua am tinh. Khong dot them chu ky.")